# INFORME DE GUÍA PRÁCTICA - APRENDIZAJE PRÁCTICO EXPERIMENTAL (APE)
## Clasificación de Dígitos Manuscritos con MNIST (Árbol de Decisión, Random Forest y k-NN)

- **Carrera:** Software
- **Asignatura:** Inteligencia Artificial
- **Docente:** Rubén Nogales
- **Unidad de Organización Curricular:** Profesional
- **Nivel y Paralelo:** Séptimo Semestre - A

---

### 1. Configuración del Entorno y Parche de Compatibilidad
Se registran las versiones del sistema y se aplica un envoltorio seguro para el analizador ARFF de `scikit-learn` que previene el error con tipos enteros en `re.search` (Python 3.14+).

In [ ]:
import sys
import time
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

_orig_re_search = re.search
def _safe_re_search(pattern, string, flags=0):
    if not isinstance(string, (str, bytes, bytearray)):
        string = str(string)
    return _orig_re_search(pattern, string, flags)
re.search = _safe_re_search

from pandas.core.arrays.categorical import Categorical
_orig_rename_categories = Categorical.rename_categories
def _safe_rename_categories(self, new_categories, *args, **kwargs):
    if callable(new_categories):
        orig_func = new_categories
        def _wrapped(val):
            try:
                return orig_func(val)
            except (TypeError, AttributeError):
                return str(val)
        return _orig_rename_categories(self, _wrapped, *args, **kwargs)
    return _orig_rename_categories(self, new_categories, *args, **kwargs)
Categorical.rename_categories = _safe_rename_categories

import sklearn
import sklearn.datasets._arff_parser
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

print(f"Versión de Python      : {sys.version.split()[0]}")
print(f"Versión de scikit-learn: {sklearn.__version__}")

### Actividad 1. Preparación de MNIST
- Carga de `mnist_784` (OpenML ID 554).
- Normalización espacial dividiendo para 255.0 ($[0.0, 1.0]$).
- Muestreo estratificado con semilla 42 (10 000 entrenamiento, 2 000 prueba) usando `tomar_muestra`.
- Verificación de dimensiones y reporte de distribución por dígito.

In [ ]:
print("Descargando / Cargando MNIST (OpenML ID: 554)...")
X, y = fetch_openml(data_id=554, as_frame=False, return_X_y=True)

X = X.astype("float32") / 255.0
y = y.astype("int64")

def tomar_muestra(X_base, y_base, n):
    X_muestra, _, y_muestra, _ = train_test_split(
        X_base, y_base, train_size=n, stratify=y_base, random_state=42
    )
    return X_muestra, y_muestra

X_ent, y_ent = tomar_muestra(X[:60000], y[:60000], 10000)
X_pru, y_pru = tomar_muestra(X[60000:], y[60000:], 2000)

print(f"Dimensiones X_ent: {X_ent.shape} | y_ent: {y_ent.shape}")
print(f"Dimensiones X_pru: {X_pru.shape}  | y_pru: {y_pru.shape}\n")

df_distribucion = pd.DataFrame({
    "Dígito": list(range(10)),
    "Frecuencia Entrenamiento": np.bincount(y_ent),
    "Proporción Ent. (%)": (np.bincount(y_ent) / len(y_ent)) * 100.0,
    "Frecuencia Prueba": np.bincount(y_pru),
    "Proporción Pru. (%)": (np.bincount(y_pru) / len(y_pru)) * 100.0,
})
display(df_distribucion)

#### Visualización de Muestras de Entrenamiento
Visualización de 5 ejemplos representativos con sus etiquetas reales.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(10, 2.5))
for i in range(5):
    axes[i].imshow(X_ent[i].reshape(28, 28), cmap="gray")
    axes[i].set_title(f"Dígito: {y_ent[i]}", fontsize=11, fontweight="bold")
    axes[i].axis("off")
plt.tight_layout()
plt.savefig("figura1_muestras_mnist.png", dpi=300)
plt.show()

### Actividad 2. Entrenamiento de los Tres Modelos
Se configuran los modelos de acuerdo a la **Tabla 1** de la guía práctica:
- **Árbol de decisión:** `max_depth=15, random_state=42`
- **Random Forest:** `n_estimators=100, max_depth=15, random_state=42`
- **k-NN:** `n_neighbors=5, metric='euclidean', weights='uniform'`

*Se añade `n_jobs=-1` en Random Forest y k-NN para aprovechar todos los núcleos de la CPU sin alterar los parámetros matemáticos.*

In [ ]:
modelos = {
    "Árbol de decisión": DecisionTreeClassifier(
        max_depth=15,
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        max_depth=15,
        random_state=42,
        n_jobs=-1
    ),
    "k-NN": KNeighborsClassifier(
        n_neighbors=5,
        metric="euclidean",
        weights="uniform",
        n_jobs=-1
    )
}

tiempos_entrenamiento = {}
for nombre, modelo in modelos.items():
    t0 = time.time()
    modelo.fit(X_ent, y_ent)
    duracion = time.time() - t0
    tiempos_entrenamiento[nombre] = duracion
    print(f"✓ {nombre} ajustado en {duracion:.2f} segundos.")

### Actividad 3. Evaluación y Cálculo de Métricas (Tabla 2)
- Inferencia separada en entrenamiento (`X_ent`) y prueba (`X_pru`).
- Cálculo de Exactitud (%) y Error de clasificación (%).
- Cálculo de la Brecha en puntos porcentuales: $\text{Exactitud}_{ent} - \text{Exactitud}_{pru}$.
- Verificación formal de que $\text{Exactitud} + \text{Error} = 100.00\%$.

In [ ]:
filas_tabla_2 = []
predicciones_ent = {}
predicciones_pru = {}

for nombre, modelo in modelos.items():
    y_pred_ent = modelo.predict(X_ent)
    y_pred_pru = modelo.predict(X_pru)
    
    predicciones_ent[nombre] = y_pred_ent
    predicciones_pru[nombre] = y_pred_pru
    
    acc_ent = accuracy_score(y_ent, y_pred_ent) * 100.0
    err_ent = 100.0 - acc_ent
    acc_pru = accuracy_score(y_pru, y_pred_pru) * 100.0
    err_pru = 100.0 - acc_pru
    brecha = acc_ent - acc_pru
    
    filas_tabla_2.append({
        "Modelo": nombre,
        "Exactitud ent. (%)": round(acc_ent, 2),
        "Error ent. (%)": round(err_ent, 2),
        "Exactitud prueba (%)": round(acc_pru, 2),
        "Error prueba (%)": round(err_pru, 2),
        "Brecha (p. p.)": round(brecha, 2)
    })

df_tabla_2 = pd.DataFrame(filas_tabla_2)

for _, r in df_tabla_2.iterrows():
    assert np.isclose(r["Exactitud ent. (%)"] + r["Error ent. (%)"], 100.0)
    assert np.isclose(r["Exactitud prueba (%)"] + r["Error prueba (%)"], 100.0)

print("TABLA 2. RESULTADOS DE ENTRENAMIENTO Y PRUEBA (Formato Oficial):")
display(df_tabla_2)

### Actividad 4. Matrices de Confusión y Análisis de Confusiones Críticas
- Generación de las tres matrices de confusión de prueba con clases de 0 a 9 en ambos ejes.
- Identificación automática de la confusión fuera de la diagonal más frecuente por modelo.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
matrices_cm = {}

for idx, (nombre, y_pred) in enumerate(predicciones_pru.items()):
    cm = confusion_matrix(y_pru, y_pred, labels=list(range(10)))
    matrices_cm[nombre] = cm
    
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=list(range(10)))
    disp.plot(ax=axes[idx], cmap="Blues", colorbar=False, values_format="d")
    axes[idx].set_title(f"{nombre}", fontsize=12, fontweight="bold")
    axes[idx].set_xlabel("Predicción")
    axes[idx].set_ylabel("Etiqueta Real" if idx == 0 else "")

plt.tight_layout()
plt.savefig("figura2_matrices_confusion.png", dpi=300)
plt.show()

print("-" * 80)
print("CONFUSIONES MÁS FRECUENTES FUERA DE LA DIAGONAL (Sección 2.7):")
print("-" * 80)
for nombre, cm in matrices_cm.items():
    cm_sin_diag = cm.copy()
    np.fill_diagonal(cm_sin_diag, 0)
    idx_max = np.argmax(cm_sin_diag)
    real_f, pred_f = np.unravel_index(idx_max, cm_sin_diag.shape)
    print(f"[{nombre}] Confusión máxima: Dígito {real_f} clasificado como {pred_f} -> ({cm_sin_diag[real_f, pred_f]} casos)")

### Inspección de Imagen de Prueba Mal Clasificada (Figura 3)
Búsqueda y visualización de una imagen del conjunto de prueba donde al menos un modelo falle, reportando la etiqueta real y las tres predicciones emitidas.

In [ ]:
indices_fallos = []
for i in range(len(y_pru)):
    fallo = (
        predicciones_pru["Árbol de decisión"][i] != y_pru[i] or
        predicciones_pru["Random Forest"][i] != y_pru[i] or
        predicciones_pru["k-NN"][i] != y_pru[i]
    )
    if fallo:
        indices_fallos.append(i)

idx_ejemplo = indices_fallos[0]
img_ejemplo = X_pru[idx_ejemplo].reshape(28, 28)
real_val = y_pru[idx_ejemplo]
pred_dt = predicciones_pru["Árbol de decisión"][idx_ejemplo]
pred_rf = predicciones_pru["Random Forest"][idx_ejemplo]
pred_knn = predicciones_pru["k-NN"][idx_ejemplo]

fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(img_ejemplo, cmap="gray")
titulo_error = (
    f"Etiqueta Real: {real_val}\n"
    f"Pred. Árbol: {pred_dt} | Pred. RF: {pred_rf} | Pred. k-NN: {pred_knn}"
)
ax.set_title(titulo_error, fontsize=10, fontweight="bold", pad=10)
ax.axis("off")
plt.tight_layout()
plt.savefig("figura3_caso_mal_clasificado.png", dpi=300)
plt.show()